# Day 2 — Linear & Logistic Regression

Implement from scratch with numpy, then verify against scikit-learn.


## 1. Linear regression from scratch (gradient descent)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)
X = rng.uniform(-3, 3, size=(200, 1))
y = 2.5 * X.squeeze() + 1.0 + rng.normal(0, 1, size=200)

def fit_linreg_gd(X, y, lr=0.05, epochs=500):
    n, d = X.shape
    Xb = np.hstack([np.ones((n, 1)), X])  # add bias
    w = np.zeros(d + 1)
    history = []
    for _ in range(epochs):
        y_hat = Xb @ w
        err = y_hat - y
        grad = (2 / n) * Xb.T @ err
        w -= lr * grad
        history.append(np.mean(err ** 2))
    return w, history

w, hist = fit_linreg_gd(X, y)
print('intercept, slope =', w)
plt.plot(hist); plt.xlabel('epoch'); plt.ylabel('MSE'); plt.title('GD convergence');


## 2. Verify with scikit-learn

In [ ]:
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error

data = fetch_california_housing(as_frame=True)
X, y = data.data, data.target
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=0)

sc = StandardScaler().fit(X_tr)
X_tr_s, X_te_s = sc.transform(X_tr), sc.transform(X_te)

for name, model in [('LR', LinearRegression()), ('Ridge', Ridge(alpha=1.0)), ('Lasso', Lasso(alpha=0.01))]:
    model.fit(X_tr_s, y_tr)
    rmse = mean_squared_error(y_te, model.predict(X_te_s), squared=False)
    print(f'{name:6s}  test RMSE = {rmse:.3f}')


## 3. Logistic regression from scratch

In [ ]:
from sklearn.datasets import make_moons
from sklearn.linear_model import LogisticRegression

X, y = make_moons(n_samples=400, noise=0.25, random_state=0)

def sigmoid(z): return 1 / (1 + np.exp(-z))

def fit_logreg(X, y, lr=0.1, epochs=2000):
    Xb = np.hstack([np.ones((len(X), 1)), X])
    w = np.zeros(Xb.shape[1])
    for _ in range(epochs):
        p = sigmoid(Xb @ w)
        grad = Xb.T @ (p - y) / len(X)
        w -= lr * grad
    return w

w = fit_logreg(X, y)
print('weights:', w)

# Compare
skl = LogisticRegression().fit(X, y)
print('sklearn:', skl.intercept_, skl.coef_)


## 4. Plot the decision boundary

In [ ]:
xx, yy = np.meshgrid(np.linspace(-1.5, 2.5, 200), np.linspace(-1, 1.5, 200))
grid = np.c_[xx.ravel(), yy.ravel()]
gridb = np.hstack([np.ones((len(grid), 1)), grid])
probs = sigmoid(gridb @ w).reshape(xx.shape)

plt.contourf(xx, yy, probs, levels=20, cmap='RdBu', alpha=0.6)
plt.scatter(X[:, 0], X[:, 1], c=y, edgecolors='k', cmap='RdBu')
plt.title('Logistic regression decision boundary');


### Exercises
- Add L2 regularization to the from-scratch logistic regression.
- Try polynomial features and re-fit on `make_moons`.
- Implement mini-batch SGD instead of full-batch GD.
